# Vector Stores

## FAISS

Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("./data/speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [2]:
docs

[Document(metadata={'source': './data/speech.txt'}, page_content='"Shakespeare" redirects here. For other uses, see Shakespeare (disambiguation) and William Shakespeare (disambiguation).\nWilliam Shakespeare\n\nThe Chandos portrait, likely depicting Shakespeare, c.\u20091611\nBorn\tc.\u200923 April 1564\nStratford-upon-Avon, Warwickshire, England\nDied\t23 April 1616 (aged 51–52)\nStratford-upon-Avon, Warwickshire, England\nResting place\tChurch of the Holy Trinity, Stratford-upon-Avon\nOccupations\t\nPlaywrightpoetactor\nYears active\tc.\u20091585–1613\nEra\t\nElizabethanJacobean\nOrganisations\t\nLord Chamberlain\'s MenKing\'s Men\nWorks\tShakespeare bibliography\nMovement\tEnglish Renaissance\nSpouse\tAnne Hathaway \u200b(m. 1582)\u200b\nChildren\t\nSusanna Hall\nHamnet Shakespeare\nJudith Quiney\nParents\t\nJohn Shakespeare\nMary Arden\nWriting career\nLanguage\tEarly Modern English\nGenres\t\nPlay (comedyhistorytragedy)\nPoetry (sonnetnarrative poemepitaph)\nSignature'),
 Document

In [3]:
embeddings = OllamaEmbeddings(
    model="gemma:2b",
)
db = FAISS.from_documents(docs, embeddings)
db

/var/folders/d3/399k69l53y170wj7fmjtwd240000gn/T/ipykernel_4694/601158098.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [5]:
### querying
query = "which year was Shakespeare born?"
docs = db.similarity_search(query)
docs[0].page_content

'William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'

### As a Retriever

We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [6]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'

### Similarity Search with score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [7]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='bfcc65ef-b08c-4780-a916-87732c94084c', metadata={'source': './data/speech.txt'}, page_content='William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'),
  np.float32(2051.1123)),
 (Document(id='9e498fe5-c2e8-4ce6-907a-99fb1136bbe9', metadata={'source': './data/speech.txt'}, page_content="Shakespeare was

In [8]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.06292594969272614,
 -0.559682309627533,
 1.0592645406723022,
 2.785205364227295,
 0.866852343082428,
 2.268808603286743,
 0.765883207321167,
 0.5405739545822144,
 2.067233085632324,
 -0.7072830200195312,
 2.1842408180236816,
 -1.1488840579986572,
 0.41459640860557556,
 0.5039002895355225,
 0.2600235641002655,
 -0.029123730957508087,
 0.3085443079471588,
 0.6524559855461121,
 0.023835811764001846,
 -1.8892971277236938,
 0.7626821398735046,
 0.0036367475986480713,
 0.28505006432533264,
 -1.314877986907959,
 -0.7208566665649414,
 -0.40052077174186707,
 0.28780311346054077,
 -1.696617603302002,
 0.07514800131320953,
 -3.1598191261291504,
 -0.3485454320907593,
 -0.14743900299072266,
 1.3716471195220947,
 -0.6555835604667664,
 0.08630684018135071,
 -0.01091464888304472,
 -1.5865601301193237,
 -0.3793206512928009,
 1.1144582033157349,
 -0.46918103098869324,
 -0.30319225788116455,
 0.6611450910568237,
 0.1028188169002533,
 -0.6981160044670105,
 1.1287752389907837,
 -0.28967317938804626,
 1

In [9]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='bfcc65ef-b08c-4780-a916-87732c94084c', metadata={'source': './data/speech.txt'}, page_content='William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'),
 Document(id='9e498fe5-c2e8-4ce6-907a-99fb1136bbe9', metadata={'source': './data/speech.txt'}, page_content="Shakespeare was the son of John Shakespeare

In [11]:
### Saving And Loading
db.save_local("./indexes/faiss_index")

In [12]:
new_db = FAISS.load_local(
    "./indexes/faiss_index", embeddings, allow_dangerous_deserialization=True
)
docs = new_db.similarity_search(query)

In [13]:
docs

[Document(id='bfcc65ef-b08c-4780-a916-87732c94084c', metadata={'source': './data/speech.txt'}, page_content='William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'),
 Document(id='9e498fe5-c2e8-4ce6-907a-99fb1136bbe9', metadata={'source': './data/speech.txt'}, page_content="Shakespeare was the son of John Shakespeare

## Chroma

Chroma is a **AI-native open-source vector database** focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

https://python.langchain.com/v0.2/docs/integrations/vectorstores/

In [14]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
loader = TextLoader("./data/speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [18]:
embeddings = OllamaEmbeddings(
    model="gemma:2b",
)
vectordb = Chroma.from_documents(documents=docs, embedding=embeddings)
vectordb

In [19]:
## query it
docs = vectordb.similarity_search(query)
docs[0].page_content

'William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'

In [20]:
## Saving to the disk
vectordb = Chroma.from_documents(
    documents=docs, embedding=embeddings, persist_directory="./indexes/chroma_db"
)


In [22]:
# load from disk
db2 = Chroma(persist_directory="./indexes/chroma_db", embedding_function=embeddings)
docs = db2.similarity_search(query)
print(docs[0].page_content)


William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world's pre-eminent dramatist. He is often called England's national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.


In [23]:
## similarity Search With Score
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='2232c3fb-dda5-4302-aff0-6eb55d48f3b0', metadata={'source': './data/speech.txt'}, page_content='William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'),
  2051.1123046875),
 (Document(id='df4a9ece-38cd-4554-ba06-f193a2392e95', metadata={'source': './data/speech.txt'}, page_content="Shakespeare was the s

In [24]:
### Retriever option
retriever = vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'William Shakespeare[a] (c. 23 April 1564[b] – 23 April 1616)[c] was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world\'s pre-eminent dramatist. He is often called England\'s national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaborations, consist of some 39 plays, 154 sonnets, three long narrative poems and a few other verses, some of uncertain authorship. His plays have been translated into every major living language and are performed more often than those of any other playwright. Shakespeare remains arguably the most influential writer in the English language, and his works continue to be studied and reinterpreted.'